[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/ROBT407-SVM-Labs/blob/main/student/203_kernel_svm.ipynb)

# ROBT 407 · Kernel SVMs: explicit features, implicit geometry, and RBFs

**Student workbook | Python 3 | Core: 90 minutes | Deepening work + mini-project: 2–3 hours**

Verify a polynomial feature map; implement polynomial and RBF kernels; compare explicit-feature and precomputed-kernel predictions; investigate gamma and Gram-matrix validity.

Use the cycle **predict → run → inspect → explain**. Write predictions before running experiments. Complete the TODO functions and rerun their checks. Synthetic experiments and bundled scikit-learn datasets are available locally; no dataset downloads are needed. Core activities follow the lecture; deepening labs and mini-projects extend it.

## Lecture alignment and course context
Source: **203_Kernel-SVMs - Copy - Copy.pdf**. PDF pages 3–13: nonlinear transformations and feature growth; 14–20: kernel trick and polynomials; 21–26: Gaussian kernel, comparisons and kernel validity. Page numbers refer to physical PDF pages, starting at 1, rather than the slide footer.

Course context: ROBT 407, Fall 2026; these activities support theoretical reasoning (CLO1), pipeline design and evaluation (CLO2), and Python implementation (CLO3). Textbook context: *Learning From Data*, Yaser S. Abu-Mostafa, Malik Magdon-Ismail, Hsuan-Tien Lin. No textbook excerpt was supplied; these are original explanations and exercises based on the uploaded slides, with no claimed textbook page mapping.

The lecture-specific quiz checks understanding of slide examples; it does not verify whether someone watched a video.

**Notation:** rows are examples; labels are −1 and +1 unless stated otherwise. Bias is separate from w. Margin means one-sided geometric margin unless explicitly called full width.

## Setup
Run cells in order in JupyterLab, Jupyter Notebook, or Google Colab with a Python 3 kernel. Required packages: NumPy, SciPy, Matplotlib, scikit-learn; ipywidgets is optional. Static plots and editable function calls remain available without widgets. Live sliders are opt-in: set ENABLE_WIDGETS=True in Setup and rerun an experiment cell. Static figures are the default for predictable batch execution. Student notebooks intentionally leave implementations incomplete; their checks print reminders until completed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.svm import SVC
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from IPython.display import display
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({'figure.figsize': (7, 5), 'axes.grid': True, 'font.size': 11})
SEED = 407

def boundary(model, X, y, title='', ax=None):
    if ax is None: _, ax = plt.subplots()
    low, high = X.min(axis=0)-.6, X.max(axis=0)+.6
    xx, yy = np.meshgrid(np.linspace(low[0],high[0],160), np.linspace(low[1],high[1],160))
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=[-np.inf,0,np.inf], colors=['#e7edff','#ffe9da'], alpha=.8)
    ax.contour(xx, yy, zz, levels=[-1,0,1], colors='black', linestyles=['--','-','--'])
    ax.scatter(X[:,0],X[:,1],c=y,cmap='coolwarm',edgecolors='black',s=35)
    if hasattr(model,'support_'):
        sv = X[model.support_]
        ax.scatter(sv[:,0],sv[:,1],s=115,facecolors='none',edgecolors='#208050',linewidths=1.5,label='Support vectors')
        ax.legend(loc='best')
    ax.set(xlabel='$x_1$',ylabel='$x_2$',title=title)
    return ax

ENABLE_WIDGETS = False  # Set True and rerun the experiment cells for live sliders.

def optional_slider(fn, name, lo, hi, value, step):
    if not ENABLE_WIDGETS:
        print("To enable live sliders, set ENABLE_WIDGETS = True in Setup; direct function calls also work.")
        return
    # Static experiments remain available if this frontend cannot display widgets.
    try:
        import ipywidgets as widgets
        display(widgets.interactive(fn, **{name: widgets.FloatSlider(min=lo,max=hi,value=value,step=step,continuous_update=False)}))
    except ImportError:
        print('Widgets unavailable; call the experiment function with a new value.')

## 1. Linear in feature space, nonlinear in input space (10 min)
With $z=\Phi(x)$ the dual only needs inner products:
$$K(x,x')=\Phi(x)^T\Phi(x'),\qquad
f(x)=\sum_{n\in SV}\alpha_n y_n K(x_n,x)+b.$$
A kernel avoids constructing every transformed coordinate. It does not eliminate the training Gram matrix or make training independent of $N$.

For two input features, an exact map for $K(x,x')=(1+x^Tx')^2$ is
$$\Phi(x)=(1,\sqrt2x_1,\sqrt2x_2,x_1^2,\sqrt2x_1x_2,x_2^2).$$
The square-root factors matter: unweighted polynomial features span similar functions but induce a different norm and hence a different margin geometry. The slide kernel $1+t+t^2$ is also different from $(1+t)^2=1+2t+t^2$.

### Implement A: an exact quadratic feature map (10 min)
Construct the six columns of $\Phi$ above. Verify the identity on random inputs, not just the training data.

In [ ]:
def phi2(X):
    # TODO: shape (N,2) -> shape (N,6)
    return None

In [ ]:
rng=np.random.default_rng(SEED)
A=rng.normal(size=(7,2));B=rng.normal(size=(5,2))
features=phi2(A)
if features is None: print('Complete Implement A, then rerun.')
else:
    assert features.shape==(7,6)
    np.testing.assert_allclose(features@phi2(B).T,(1+A@B.T)**2,atol=1e-10)
    print('Exact feature-map identity passed.')

**Your reasoning:** write your prediction, observations and explanation here.

## 2. Implement the similarities (10 min)
$$K_{poly}(x,x')=(\zeta+\gamma x^Tx')^Q,\quad \gamma>0,\ \zeta\ge0,\ Q\in\mathbb N,$$
$$K_{RBF}(x,x')=\exp(-\gamma\|x-x'\|^2),\quad\gamma>0.$$
Use $\|a-b\|^2=\|a\|^2+\|b\|^2-2a^Tb$ to avoid allocating an $N\times M\times d$ difference tensor. Clip tiny negative squared distances caused by roundoff.

Large $\gamma$ makes Gaussian similarities more local. It can make a more variable boundary, but its effect depends on $C$, scaling and the data. It is not a guarantee of overfitting.

### Implement B: vectorized RBF kernel (10 min)
Return a rectangular query-to-training matrix. Do not use a library kernel implementation.

In [ ]:
def rbf_kernel(A,B,gamma):
    # TODO: compute pairwise squared distances, then exponentiate.
    return None

In [ ]:
K=rbf_kernel(A,B,.7)
if K is None: print('Complete Implement B, then rerun.')
else:
    expected=np.exp(-.7*((A[:,None,:]-B[None,:,:])**2).sum(axis=2))
    np.testing.assert_allclose(K,expected,atol=1e-12)
    np.testing.assert_allclose(np.diag(rbf_kernel(A,A,.7)),1,atol=1e-12)
    assert K.shape==(7,5)
    print('RBF kernel checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

## 3. Explicit map versus kernel trick (10 min)
Both models below use the same C and exactly the same inner products. Predict whether their scores should agree. This section waits for Implement A; complete that task before running it. The demonstrations use finite-C soft-margin SVC, anticipating the next lecture.

In [ ]:
X,y01=make_circles(n_samples=160,factor=.4,noise=.06,random_state=SEED)
y=2*y01-1
Z=phi2(X)
if Z is None:
    print('Complete Implement A to run the explicit-versus-kernel experiment.')
else:
    explicit=SVC(kernel='linear',C=10,tol=1e-9).fit(Z,y)
    implicit=SVC(kernel='precomputed',C=10,tol=1e-9).fit((1+X@X.T)**2,y)
    Xq=np.random.default_rng(SEED+1).normal(size=(60,2))
    s1=explicit.decision_function(phi2(Xq))
    s2=implicit.decision_function((1+Xq@X.T)**2)
    print('maximum score discrepancy:',np.max(np.abs(s1-s2)))
    np.testing.assert_allclose(s1,s2,atol=2e-4)

## 4. Predict, run, explain: the RBF width (10 min)
Hold C fixed. Compare three gamma values before using the slider. Record training accuracy, number of support vectors and boundary shape. These are exploratory training observations; model selection with held-out data comes next class.

In [ ]:
Xm,ym01=make_moons(n_samples=130,noise=.2,random_state=SEED)
ym=2*ym01-1
fig,axs=plt.subplots(1,3,figsize=(15,4))
for gamma,ax in zip([.1,1,30],axs):
    m=SVC(C=10,gamma=gamma).fit(Xm,ym)
    boundary(m,Xm,ym,f'gamma={gamma}; train={m.score(Xm,ym):.2f}; SV={len(m.support_)}',ax)
plt.tight_layout();plt.show()
def gamma_experiment(log_gamma=0.):
    m=SVC(C=10,gamma=10**log_gamma).fit(Xm,ym)
    boundary(m,Xm,ym,f'gamma={10**log_gamma:.3g}, C=10');plt.show()
optional_slider(gamma_experiment,'log_gamma',-2.,2.,0.,.2)

## 5. Is any similarity a kernel? (10 min)
A real-valued kernel must be symmetric and have a positive semidefinite Gram matrix for **every** finite collection of inputs:
$$v^TKv\ge0\quad\text{for every vector }v.$$
A nonnegative smallest eigenvalue on one dataset is evidence about that matrix, not proof that the function is valid everywhere. Negative entries alone do not invalidate a kernel.

In [ ]:
def gram_diagnostics(K):
    symmetric=np.allclose(K,K.T,atol=1e-10)
    eigenvalues=np.linalg.eigvalsh((K+K.T)/2)
    return symmetric,eigenvalues.min()
valid=np.exp(-((A[:,None,:]-A[None,:,:])**2).sum(axis=2))
invalid=np.array([[1.,2.],[2.,1.]])
print('RBF Gram (symmetric, minimum eigenvalue):',gram_diagnostics(valid))
print('Counterexample:',gram_diagnostics(invalid))
v=np.array([1.,-1.]);print('v^T K v for counterexample:',v@invalid@v)

### Reflection and submission
Explain why feature rescaling can change an SVM despite leaving the span of candidate decision functions unchanged. Give one reason to start with a linear kernel. Submit both implementations, the numerical feature-map identity, three gamma observations, and the invalid-kernel counterexample.

## Deepening lab A · Geometry is chosen by the kernel (15 min)
For a positive semidefinite $K$, $v^TKv\ge0$. If $K_1,K_2$ are valid kernels and $a,b\ge0$, then $aK_1+bK_2$ is valid, since its quadratic form is the corresponding sum of nonnegative terms. One feature map is the concatenation $(\sqrt a\Phi_1,\sqrt b\Phi_2)$.

**Derive:** why does a negative combination weight break this general proof? Why might a combined linear–RBF kernel be useful for a mixture of global trends and local structure?

### Implement C · Combine valid kernels
Mix a linear kernel and an RBF kernel using nonnegative weights. Reuse your RBF implementation. Validate the parameters; returning a symmetric matrix alone is not sufficient.

In [ ]:
def mixed_kernel(A,B,linear_weight,rbf_weight,gamma):
    # TODO: nonnegative weighted sum of the two kernels.
    return None

In [ ]:
mix=mixed_kernel(A,A,.3,.7,1.)
if mix is None: print('Complete Implement C (and B).')
else:
    np.testing.assert_allclose(mix,mix.T,atol=1e-12)
    assert np.linalg.eigvalsh(mix).min()>-1e-10
    np.testing.assert_allclose(mixed_kernel(A,B,1.,0.,1.),A@B.T)
    try: mixed_kernel(A,B,-1.,1.,1.)
    except ValueError: pass
    else: raise AssertionError('Reject negative weights')
    print('Kernel combination checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

In [ ]:
# Order by class to expose blocks in the Gram matrices.
order=np.argsort(ym);Xordered=Xm[order]
fig,axs=plt.subplots(1,3,figsize=(14,4))
for gamma,ax in zip([.1,1.,30.],axs):
    squared=((Xordered[:,None,:]-Xordered[None,:,:])**2).sum(axis=2)
    im=ax.imshow(np.exp(-gamma*squared),vmin=0,vmax=1,cmap='magma')
    ax.set(title=f'RBF Gram, gamma={gamma}',xlabel='Training example',ylabel='Training example')
fig.colorbar(im,ax=list(axs),label='Similarity',shrink=.7);plt.show()

## Deepening lab B · Approximate a kernel with random features (20 min)
The RBF kernel can be approximated with explicit randomized features:
$$z_j(x)=\sqrt{2/D}\cos(\omega_j^Tx+\beta_j),\qquad
\omega_j\sim\mathcal N(0,2\gamma I),\quad\beta_j\sim U(0,2\pi).$$
Then $z(x)^Tz(x')$ approximates $\exp(-\gamma\|x-x'\|^2)$. This is an optional extension beyond the supplied slide details. More features tend to improve the approximation on average, but a single random realization need not improve monotonically. The same sampled frequencies and phases must be reused for training and test inputs.

### Implement D · Random Fourier features
Return transformed X and the sampled frequencies and phases. Allow an existing pair of parameters to be reused. Validate your implementation by comparing average Gram approximation errors across multiple seeds.

In [ ]:
def rff(X,D,gamma,seed=407,parameters=None):
    # TODO: return (Z, (frequencies, phases)).
    return None

In [ ]:
approx=rff(A,100,1.)
if approx is None: print('Complete Implement D.')
else:
    Z,params=approx
    np.testing.assert_allclose(rff(A,100,1.,parameters=params)[0],Z)
    sizes=[16,64,256,1024];average=[];spread=[]
    exact=np.exp(-((A[:,None,:]-A[None,:,:])**2).sum(axis=2))
    for D in sizes:
        errors=[]
        for seed in range(5):
            Z,_=rff(A,D,1.,seed=seed)
            errors.append(np.linalg.norm(Z@Z.T-exact)/np.linalg.norm(exact))
        average.append(np.mean(errors));spread.append(np.std(errors))
    assert average[-1]<average[0]
    plt.errorbar(sizes,average,yerr=spread,marker='o',capsize=4)
    plt.xscale('log');plt.xlabel('Number of random features');plt.ylabel('Relative Gram error');plt.title('Approximation quality across five seeds');plt.show()

**Your reasoning:** write your prediction, observations and explanation here.

### Implement E · Kernel prediction with support vectors only
Implement scores from a kernel callback without constructing explicit transformed coordinates. The supplied multipliers are unsigned. Select support vectors using a tolerance and preserve query/training orientation.

In [ ]:
def kernel_scores(Xq,Xtrain,y,alpha,b,kernel,tol=1e-7):
    # TODO: retain support vectors, compute their contributions, add b.
    return None

In [ ]:
test_model=SVC(C=2,gamma=.8,tol=1e-9).fit(Xm,ym)
a_full=np.zeros(len(ym));a_full[test_model.support_]=np.abs(test_model.dual_coef_[0])
Kfun=lambda A,B:np.exp(-.8*((A[:,None,:]-B[None,:,:])**2).sum(axis=2))
query=np.array([[-1.,0.],[0.,0.],[1.,.2],[2.,1.]])
custom=kernel_scores(query,Xm,ym,a_full,test_model.intercept_[0],Kfun)
if custom is None: print('Complete Implement E.')
else:
    np.testing.assert_allclose(custom,test_model.decision_function(query),atol=1e-6)
    contributions=Kfun(query,Xm[test_model.support_])*test_model.dual_coef_[0]
    fig,ax=plt.subplots();im=ax.imshow(contributions,aspect='auto',cmap='coolwarm')
    ax.set(xlabel='Support vector',ylabel='Query point',title='Signed support-vector contributions to the score')
    fig.colorbar(im,ax=ax);plt.show();print('Kernel prediction agreement passed.')

**Your reasoning:** write your prediction, observations and explanation here.

## Real-data mini-project · Handwritten digit recognition with three kernel families (45–75 min)
1,797 handwritten digit images, each 8 by 8 pixels, with ten classes and 64 numerical inputs. This bundled dataset is distinct from MNIST.

**Before fitting:** write a hypothesis about the useful kernel or regularization strength, the expected errors and the effect of scaling.

**Search budget:** Linear: tune C, 3 candidates. RBF: tune C and gamma, 9 candidates. Polynomial: tune C, gamma, degree and coef0, 16 candidates. Kernel family is a categorical model choice. Total: 28 candidates, 140 five-fold fits and one overall refit.

Use a stratified 80/20 development/test split. Hyperparameter selection uses **five-fold stratified cross-validation** on development data only. Fit scaling inside each fold through a pipeline. Reserve test labels until selection and refitting are finished. These datasets are bundled with scikit-learn; the loaders need no dataset download. Small benchmark results demonstrate a workflow rather than establish deployment performance.

### Understand the model-selection choices
| Choice | Effect | Applicable kernels |
|---|---|---|
| $C>0$ | Penalty per unit margin violation in the sum-loss objective | Linear, polynomial, RBF |
| $\gamma>0$ | Input scale in similarities | Polynomial, RBF |
| degree | Polynomial power | Polynomial |
| coef0 | Additive polynomial constant | Polynomial |
| kernel | Similarity family and feature geometry | Family comparison |

Distinguish hyperparameter **names**, candidate **values** per name, model **combinations** and total fold **fits**. Conditional grids add branch workloads instead of multiplying irrelevant parameters into every family. Five CV training/validation rotations are performed for each candidate; the winner is refitted on all development data.

SVC handles multiclass tasks using pairwise classifiers. Its packed multiclass coefficient array is not the single binary alpha vector used in earlier derivations.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import ParameterGrid, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, balanced_accuracy_score
import pandas as pd
project_data=load_digits()
project_X=project_data.data.copy();project_y=project_data.target.copy()
project_names=np.asarray(project_data.target_names).astype(str)

X_develop,X_holdout,y_develop,y_holdout=train_test_split(project_X,project_y,test_size=.2,stratify=project_y,random_state=SEED+10)
project_cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED+11)
project_grid=[{'svc__kernel':['linear'],'svc__C':[.1,1,10]}, {'svc__kernel':['rbf'],'svc__C':[.1,1,10],'svc__gamma':[.001,.01,.1]}, {'svc__kernel':['poly'],'svc__C':[.1,1],'svc__gamma':[.001,.01],'svc__degree':[2,3],'svc__coef0':[0,1]}]
project_candidates=len(list(ParameterGrid(project_grid)))
print('Data shape and class counts:',project_X.shape,np.unique(project_y,return_counts=True))
print('Development/test sizes:',len(y_develop),len(y_holdout))
print('Candidates, fold fits, fits including refit:',project_candidates,5*project_candidates,5*project_candidates+1)
assert project_cv.n_splits==5
visits=np.zeros(len(y_develop),dtype=int)
for training,validation in project_cv.split(X_develop,y_develop):
    assert len(np.intersect1d(training,validation))==0
    visits[validation]+=1
assert np.all(visits==1)

### Visual audit before fitting
Inspect development data and class counts. Any exploratory scaling below is separate from the estimator; the CV pipeline still learns its own scaler separately within every fold. For digit images, inspect pixels rather than just flattened vectors.

In [ ]:
fig,axs=plt.subplots(2,10,figsize=(14,4))
for label in range(10):
    chosen=np.flatnonzero(y_develop==label)[:2]
    for row,index in enumerate(chosen):
        axs[row,label].imshow(X_develop[index].reshape(8,8),cmap='gray_r',vmin=0,vmax=16)
        axs[row,label].set_title(f'Digit {label}');axs[row,label].axis('off')
plt.suptitle('Development images: variation within classes');plt.tight_layout();plt.show()

### Implement F · Construct a leakage-safe five-fold search
Return an **unfitted** GridSearchCV with a StandardScaler/SVC pipeline named scale and svc. Use the supplied splitter, accuracy scoring, refit=True, return_train_score=True, n_jobs=1 and error_score='raise'.

In [ ]:
def make_project_search(grid,splitter):
    # TODO: construct the pipeline and search.
    return None

In [ ]:
candidate_search=make_project_search(project_grid,project_cv)
if candidate_search is None: print('Complete Implement F to construct your own search.')
else:
    assert isinstance(candidate_search,GridSearchCV)
    assert isinstance(candidate_search.estimator,Pipeline)
    assert isinstance(candidate_search.estimator.named_steps['scale'],StandardScaler)
    assert isinstance(candidate_search.estimator.named_steps['svc'],SVC)
    assert candidate_search.cv.n_splits==5 and candidate_search.scoring=='accuracy'
    assert candidate_search.refit and candidate_search.return_train_score
    assert candidate_search.n_jobs==1 and candidate_search.error_score=='raise'
    assert not hasattr(candidate_search,'cv_results_')
    print('Search configuration checks passed.')

**Your reasoning:** write your prediction, observations and explanation here.

### Run the reference workflow, then compare your implementation
This reference is complete so the project can be explored before finishing Implement F. To test your own implementation, replace the reference construction with your function's returned search. The reference supplies the protocol, not your written analysis or conclusions. All tuning uses development data only.

In [ ]:
project_pipeline=Pipeline([('scale',StandardScaler()),('svc',SVC(cache_size=200))])
project_search=GridSearchCV(project_pipeline,project_grid,cv=project_cv,scoring='accuracy',
    refit=True,return_train_score=True,n_jobs=1,error_score='raise')
project_search.fit(X_develop,y_develop)
project_results=pd.DataFrame(project_search.cv_results_)
parameter_columns=[column for column in project_results if column.startswith('param_')]
summary_columns=['rank_test_score',*parameter_columns,'mean_train_score','mean_test_score','std_test_score','mean_fit_time']
display(project_results[summary_columns].sort_values('rank_test_score').head(12))
print('Selected parameters:',project_search.best_params_)
print('Selection mean CV accuracy:',project_search.best_score_)
print('Fold spread is descriptive, not an independent-sample confidence interval.')
baseline_scores=cross_val_score(DummyClassifier(strategy='most_frequent'),X_develop,y_develop,cv=project_cv,scoring='accuracy')
print('Majority baseline mean CV accuracy:',baseline_scores.mean())

In [ ]:
top=project_results.sort_values('rank_test_score').head(10).reset_index(drop=True)
fig,axs=plt.subplots(1,2,figsize=(14,4));x=np.arange(len(top))
axs[0].errorbar(x,top['mean_test_score'],yerr=top['std_test_score'],fmt='o',capsize=4,label='Validation mean +/- fold SD')
axs[0].plot(x,top['mean_train_score'],'s--',label='Mean training accuracy')
axs[0].axhline(baseline_scores.mean(),color='gray',linestyle=':',label='Majority baseline')
axs[0].set(xlabel='Candidate rank (top ten)',ylabel='Accuracy',title='Selection evidence and generalization gap');axs[0].legend(fontsize=8)
axs[1].scatter(project_results['mean_fit_time'],project_results['mean_test_score'],c=project_results['mean_train_score'],cmap='viridis',edgecolors='black')
axs[1].set(xlabel='Mean fit time per fold (seconds)',ylabel='Mean validation accuracy',title='Performance versus fitting cost')
plt.tight_layout();plt.show()

In [ ]:
rbf_results=project_results[project_results['param_svc__kernel']=='rbf']
heat=rbf_results.pivot(index='param_svc__C',columns='param_svc__gamma',values='mean_test_score').sort_index().sort_index(axis=1)
fig,ax=plt.subplots();im=ax.imshow(heat.values,origin='lower',cmap='viridis')
ax.set(xticks=np.arange(len(heat.columns)),xticklabels=heat.columns,yticks=np.arange(len(heat.index)),yticklabels=heat.index,xlabel='gamma',ylabel='C',title='RBF: five-fold development accuracy')
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):ax.text(j,i,f'{heat.iloc[i,j]:.3f}',ha='center',va='center',color='white' if heat.iloc[i,j]<np.median(heat.values) else 'black')
fig.colorbar(im,ax=ax);plt.show()

### Freeze the choice, then evaluate once on the held-out test set
Write down the selected configuration and expected failure modes before running this cell. Do not alter the grid based on these results. A new hypothesis after seeing test results requires a fresh evaluation plan; the existing test is no longer untouched.

In [ ]:
holdout_predictions=project_search.predict(X_holdout)
print('Final test accuracy:',accuracy_score(y_holdout,holdout_predictions))
print('Final test balanced accuracy:',balanced_accuracy_score(y_holdout,holdout_predictions))
print(classification_report(y_holdout,holdout_predictions,target_names=project_names,zero_division=0))
fig,ax=plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay.from_predictions(y_holdout,holdout_predictions,display_labels=project_names,cmap='Blues',ax=ax,colorbar=False)
ax.set_title('Final held-out test confusion matrix');plt.xticks(rotation=45,ha='right');plt.tight_layout();plt.show()

In [ ]:
wrong=np.flatnonzero(holdout_predictions!=y_holdout)
print('Misclassified test images:',len(wrong),'out of',len(y_holdout))
fig,axs=plt.subplots(2,5,figsize=(12,5))
for ax,index in zip(axs.ravel(),wrong[:10]):
    ax.imshow(X_holdout[index].reshape(8,8),cmap='gray_r',vmin=0,vmax=16)
    ax.set_title(f'True {y_holdout[index]}, predicted {holdout_predictions[index]}');ax.axis('off')
for ax in axs.ravel()[min(10,len(wrong)):]:ax.axis('off')
plt.suptitle('Error analysis: ambiguity without retuning on test results');plt.tight_layout();plt.show()
print('Support vectors per class:',project_search.best_estimator_.named_steps['svc'].n_support_)

### Implement G · Manual five-fold selection
Implement the inner grid search without calling GridSearchCV or cross_val_score. Use ParameterGrid, clone the pipeline for every fold, fit on training indices and score validation indices. Return (best parameter dictionary, list of candidate mean scores). Keep the first candidate in a tie. This is an independent implementation of the selection protocol.

In [ ]:
def manual_grid(estimator,grid,X,y,splitter):
    # TODO: iterate candidates and fold splits; fit fresh clones and average validation scores.
    return None

In [ ]:
# A small two-candidate audit; the main project above uses the full stated grid.
audit_grid={'svc__kernel':['linear'],'svc__C':[.1,1]}
audit=manual_grid(project_pipeline,audit_grid,X_develop,y_develop,project_cv)
if audit is None: print('Complete Implement G for the independent selection challenge.')
else:
    reference=GridSearchCV(project_pipeline,audit_grid,cv=project_cv,scoring='accuracy',n_jobs=1,error_score='raise').fit(X_develop,y_develop)
    best_parameters,mean_scores=audit
    np.testing.assert_allclose(mean_scores,reference.cv_results_['mean_test_score'],atol=1e-12)
    assert best_parameters==reference.best_params_
    print('Manual five-fold grid agrees with GridSearchCV:',best_parameters)

**Your reasoning:** write your prediction, observations and explanation here.

### Submit an evidence-based mini-project report
1. Your pre-experiment hypothesis and evaluation protocol.
2. Tuned parameter names, values, candidate count and five-fold fit count. Account separately for baseline and learning-curve fits.
3. Training versus validation performance, descriptive fold spread, model complexity and fit time. Explain why a small CV difference may be inconclusive.
4. A majority-baseline comparison and per-class error analysis.
5. The frozen choice, final test confusion matrix and limitations.
6. A follow-up experiment with its own fresh evaluation plan.

**Independent challenge:** implement manual five-fold grid search with cloned pipelines, the same folds, the same scoring and development-only model selection. Compare with GridSearchCV, documenting tie-breaking, order and seeds. An instructor reference is supplied separately. An optional further project is nested CV: explain the separate roles of inner selection folds and outer evaluation folds.

Your report is evaluated for reasoning and protocol, not for obtaining the highest test accuracy.

## Lecture understanding quiz (15–20 min)

Answer before discussing with classmates. For each answer, add a one-sentence justification and point to the relevant lecture concept.

**Q1. What is the coefficient of x^T x-prime in (1+x^T x-prime)^2?**  
A: 0; B: 1; C: 2; D: 4

**Q2. Why does the exact feature map contain sqrt(2)*x1*x2?**  
Short answer

**Q3. What happens to RBF similarity between distinct fixed inputs as gamma grows?**  
A: approaches 1; B: approaches 0; C: becomes negative; D: always unchanged

**Q4. A symmetric matrix has eigenvalues -1 and 3. Is it a valid Gram matrix?**  
A: yes; B: no

**Q5. Does checking one positive semidefinite Gram matrix prove universal kernel validity?**  
A: yes; B: no

**Q6. What shape must the precomputed prediction kernel have?**  
Short answer

**Q7. When applying random Fourier features to test data, should frequencies be resampled?**  
A: yes; B: no

**Q8. Why is a nonnegative sum of valid kernels valid?**  
Short answer

**Q9. Which two numerical hyperparameters are commonly tuned for a fixed RBF SVC?**  
Short answer

**Q10. Does using a polynomial kernel eliminate training dependence on sample count?**  
A: yes; B: no

In [ ]:
# Record your answers; this cell does not grade or reveal the key.
quiz_answers = {'Q1': '', 'Q2': '', 'Q3': '', 'Q4': '', 'Q5': '', 'Q6': '', 'Q7': '', 'Q8': '', 'Q9': '', 'Q10': ''}
quiz_justifications = {}
display(quiz_answers)

## Exit ticket
In 3–5 sentences: what did your experiment show, how does the equation explain it, and what question remains? Save the notebook with your completed code, plots, quiz responses and explanations.